# 01 · 复现 AMP-BERT(训练 + 测试)

在一个 notebook 内完成原文复现:
**A. 训练** — 在 Veltri 训练集上微调 ProtBERT-BFD,得到 AMP-BERT,并保存到 Google Drive;
**B. 测试** — 从 Drive 加载模型,在外部测试集(DRAMP 的 AMP ∪ AMPEP 的 non-AMP)上评估。

训练一次后,以后新开会话可直接从 *Part B* 开始(挂载 Drive → 加载模型 → 测试),无需重训。

## 0 · 环境准备
安装一个仍带经典 BERT 分词器的 `transformers` 版本(Colab 自带的太新,无法正确加载 ProtBERT-BFD),并检查 GPU。

> 运行前请先设置:**代码执行程序 → 更改运行时类型 → GPU**。

In [ ]:
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', device)   # 期望输出 cuda

## 导入依赖

In [ ]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)

In [ ]:
# 整个 notebook 共用的两个常量
MODEL_NAME = 'Rostlab/prot_bert_bfd'   # 预训练模型(分词器与模型都用它)
MAX_LEN    = 200                        # 序列截断/补齐到的长度

## 1 · 挂载 Google Drive(用于保存/加载模型)
模型训练好后会存到 Drive,这样 **Colab 断开后也不会丢失**;以后新开会话只要挂载 Drive,就能跳过训练直接加载模型做测试。

运行下面的 cell 会弹出授权窗口,按提示登录授权即可。

> 不想用 Drive?把 `MODEL_DIR` 改成本地路径(如 `'models/amp_bert'`),并跳过挂载这一步——但断开连接后模型会丢失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 模型的保存/加载目录(Drive 路径,断线不丢)
MODEL_DIR = '/content/drive/MyDrive/amp_bert'
os.makedirs(MODEL_DIR, exist_ok=True)
print('模型目录:', MODEL_DIR)

# Part A · 训练 AMP-BERT
若你已经训练并保存过模型,可跳过 Part A,直接到 Part B 加载。

## 2 · 加载训练数据
Veltri 训练集,直接从仓库读取(无需下载/clone)。

In [ ]:
TRAIN_URL = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/all_veltri.csv'
SEED = 0   # 打乱顺序的随机种子

df = pd.read_csv(TRAIN_URL, index_col=0).sample(frac=1, random_state=SEED)
print('训练样本数:', len(df))
df.head()

## 3 · 把序列编码成模型输入
先定义数据集类,再加载分词器,最后构建 `train_dataset`。

In [ ]:
class AmpDataset(Dataset):
    """把一个 DataFrame(列:aa_seq, AMP)编码成模型输入。"""
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.seqs = list(self.df['aa_seq'])
        self.labels = list(self.df['AMP'].astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # ProtBERT 要求氨基酸之间用空格分隔
        seq = ' '.join(''.join(self.seqs[idx].split()))
        ids = self.tokenizer(seq, truncation=True, padding='max_length',
                             max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample

In [ ]:
# 加载分词器(use_fast=False 直接用 vocab.txt 的 WordPiece 分词器,避开新版本的兼容问题)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False, use_fast=False)

In [ ]:
train_dataset = AmpDataset(df, tokenizer)
print('train_dataset 大小:', len(train_dataset))

## 4 · 定义评估指标
准确率、F1、精确率、召回率、MCC、ROC-AUC。

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    m = {'accuracy': accuracy_score(labels, preds), 'f1': f1,
         'precision': precision, 'recall': recall,
         'mcc': matthews_corrcoef(labels, preds)}
    if len(np.unique(labels)) == 2:                 # AUC 需要正类概率
        z = logits - logits.max(axis=-1, keepdims=True)
        probs = np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
        m['roc_auc'] = roc_auc_score(labels, probs[:, 1])
    return m

## 5 · 定义模型
在 ProtBERT-BFD 上接一个二分类头。

In [ ]:
def model_init():   # 必须零参数(Trainer 会把 trial 传给带参数的版本)
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

## 6 · 配置训练参数
有效批大小 = 1 × 64 = 64,与论文一致。想先快速跑通,把 `num_train_epochs` 改成 1。

In [ ]:
OUTPUT_DIR = 'results/amp_bert_train'   # 训练中间产物目录

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,            # 完整复现用 15;冒烟测试用 1
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=64,  # 有效批大小 = 1 × 64
    weight_decay=0.1,
    warmup_steps=0,
    logging_steps=100,
    eval_strategy='no',
    save_strategy='no',
    fp16=True,
    seed=SEED,
    report_to='none',
)

## 7 · 开始训练
完整 15 epoch 在单卡上需要数小时。

In [ ]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

## 8 · 保存 AMP-BERT 到 Google Drive
保存到上面挂载的 `MODEL_DIR`(Drive),断线也不会丢。

In [ ]:
trainer.save_model(MODEL_DIR)
print('AMP-BERT 已保存到', MODEL_DIR)

# Part B · 测试 AMP-BERT
从 Drive 加载模型并在外部测试集上评估。**新开会话直接从这里开始**(确保已运行上面的环境准备、导入、挂载 Drive 三步)。

## 9 · 从 Google Drive 加载模型

In [ ]:
assert os.path.exists(MODEL_DIR), f'{MODEL_DIR} 不存在,请先完成 Part A 的训练与保存。'
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
print('已从 Drive 加载模型:', MODEL_DIR)

## 10 · 准备外部测试集
把 AMP 与 non-AMP 两个文件合并成完整测试集。

In [ ]:
AMP_URL    = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/veltri_dramp_cdhit_90.csv'
NONAMP_URL = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/non_amp_ampep_cdhit90.csv'

amp_df    = pd.read_csv(AMP_URL, index_col=0)
nonamp_df = pd.read_csv(NONAMP_URL, index_col=0)
test_df = pd.concat([amp_df, nonamp_df])
print('AMP:', len(amp_df), '| non-AMP:', len(nonamp_df), '| 合计:', len(test_df))
test_df.head()

In [ ]:
# 复用 Part A 定义的 AmpDataset 与 tokenizer;若单独从 Part B 运行,确保已执行第 3、4 步的 cell
test_dataset = AmpDataset(test_df, tokenizer)
print('test_dataset 大小:', len(test_dataset))

## 11 · 评估

In [ ]:
eval_args = TrainingArguments(output_dir='results/amp_bert_test',
                              per_device_eval_batch_size=56, report_to='none')
evaluator = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)
predictions, label_ids, metrics = evaluator.predict(test_dataset)
metrics

## 12 · 保存逐条预测结果

In [ ]:
PRED_CSV = 'results/external_test_predictions.csv'
os.makedirs('results', exist_ok=True)
out = test_df.copy()
out['pred'] = predictions.argmax(-1)
out.to_csv(PRED_CSV)
print('已写入', PRED_CSV)